In [7]:
# Import packages
import torch
import pandas as pd
from pathlib import Path

# Pathways 
out_dir = Path("..") / "data" / "generated"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "bs_collocation.parquet"

### Generating synthetic data

In [8]:
# Defining a function to generate synthetic data 
def generate(num_interior=10000, num_boundary=2000, S_max=150.0, T_max=1.0):
    """
    Generating synthetic data for PINN training
    """
    
    # (1) Generate the PyTorch Tensors
    S_interior = (torch.rand(num_interior, 1) * S_max).requires_grad_(True)
    tau_interior = (torch.rand(num_interior, 1) * T_max).requires_grad_(True)

    S_ic = (torch.rand(num_boundary, 1) * S_max).requires_grad_(True)
    tau_ic = torch.zeros(num_boundary, 1).requires_grad_(True)

    S_bc_zero = torch.zeros(num_boundary, 1).requires_grad_(True)
    tau_bc = (torch.rand(num_boundary, 1) * T_max).requires_grad_(True)
    
    # (2) Convert to Pandas DataFrames 
    # (Detaching from the PyTorch graph)
    df_interior = pd.DataFrame({
        'S': S_interior.detach().numpy().flatten(),
        'tau': tau_interior.detach().numpy().flatten(),
        'point_type': 'interior'
    })

    df_ic = pd.DataFrame({
        'S': S_ic.detach().numpy().flatten(),
        'tau': tau_ic.detach().numpy().flatten(),
        'point_type': 'initial_condition' # Maturity
    })

    df_bc = pd.DataFrame({
        'S': S_bc_zero.detach().numpy().flatten(),
        'tau': tau_bc.detach().numpy().flatten(),
        'point_type': 'boundary_condition' # S=0
    })

    # Combine into a neat dataset
    df_all = pd.concat([df_interior, df_ic, df_bc], ignore_index=True)

    # (3) Saving to Parquet using pathways
    df_all.to_parquet(out_path)
    print(f"Data successfully saved to {out_path}")
    print(df_all['point_type'].value_counts())

    return S_interior, tau_interior, S_ic, tau_ic, S_bc_zero, tau_bc   

In [9]:
# Running generate
S_in, tau_in, S_ic, tau_ic, S_bc, tau_bc = generate()

Data successfully saved to ..\data\generated\bs_collocation.parquet
point_type
interior              10000
initial_condition      2000
boundary_condition     2000
Name: count, dtype: int64
